# 03 Backtest Analysis

## Goal
Compare Week 4 baseline strategy artifacts with SPY and equal-weight buy-and-hold on the same date range. This notebook describes historical behavior only; it does not establish tradability or future alpha.

## Guardrails
- Factor/signal at t becomes position and PnL from t+1.
- Results include 5 bps transaction cost and 5 bps slippage per unit of turnover.
- Do not tune thresholds/windows based on these results.
- Treat survivorship bias, yfinance limitations and adjusted-close execution assumptions as material limitations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from data_pipeline.storage.local_store import BACKTESTS_DIR
from ml.backtesting.storage import list_backtests, load_backtest_result

metadata_rows = list_backtests(BACKTESTS_DIR)
assert len(metadata_rows) == 2

artifacts = {}
for metadata in metadata_rows:
    backtest_id = metadata["backtest_id"]
    artifacts[metadata["strategy_name"]] = load_backtest_result(backtest_id, BACKTESTS_DIR)

for strategy_name, (metadata, daily) in artifacts.items():
    assert list(daily.columns) == [
        "date", "gross_return", "turnover", "transaction_cost", "net_return",
        "portfolio_exposure", "equity_curve",
    ]
    assert daily["date"].is_monotonic_increasing
    assert daily["equity_curve"].gt(0).all()
    print(strategy_name, metadata["metrics"])

In [ ]:
fig, axis = plt.subplots(figsize=(12, 5))
for strategy_name, (_, daily) in artifacts.items():
    axis.plot(daily["date"], daily["equity_curve"], label=strategy_name)
axis.set_title("Week 4 strategy equity curves (net of modeled costs)")
axis.set_xlabel("Date")
axis.set_ylabel("Equity, initial = 1.0")
axis.legend()
plt.show()

In [ ]:
for strategy_name, (_, daily) in artifacts.items():
    drawdown = daily["equity_curve"] / daily["equity_curve".cummax() - 1 if "cummax" in dir(daily["equity_curve"]) else 1.0]
    drawdown_series = daily["equity_curve"] / daily["equity_curve"].cummax() - 1.0
    print(
        strategy_name,
        {
            "total_cost": daily["transaction_cost"].sum(),
            "average_turnover": daily["turnover"].mean(),
            "max_drawdown": drawdown_series.min(),
            "average_exposure": daily["portfolio_exposure"].mean(),
        },
    )